# AnimalCLEF2026 SAM3 + SIFT Graph v20260423

This notebook tests the segmentation-first local-feature idea:

- Use SAM3 text-prompted segmentation to crop/mask the animal and suppress background.
- Extract SIFT/RootSIFT features from the segmented crop.
- Build a visual-word TF-IDF shortlist so we do **not** do slow all-pairs SIFT matching.
- Run real SIFT ratio-test + affine RANSAC only on mutual shortlist edges.
- Cluster each species with a conservative graph to avoid giant false clusters.

Important: this notebook does not hardcode any Hugging Face token. If `facebook/sam3` requires access in your runtime, add `HF_TOKEN` as a Kaggle Secret or environment variable.

In [ ]:
import importlib.util, subprocess, sys

base_packages = ['opencv-python-headless', 'scikit-learn', 'tqdm', 'pillow', 'huggingface_hub', 'accelerate']
missing = []
for mod, pkg in [('cv2','opencv-python-headless'), ('sklearn','scikit-learn'), ('tqdm','tqdm'), ('PIL','pillow'), ('huggingface_hub','huggingface_hub')]:
    if importlib.util.find_spec(mod) is None:
        missing.append(pkg)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

# SAM3 is recent. If the installed Transformers does not expose Sam3Model, upgrade.
try:
    from transformers import Sam3Processor, Sam3Model
    print('[setup] installed transformers has SAM3')
except Exception:
    print('[setup] upgrading transformers for SAM3 support')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'transformers'])
    try:
        from transformers import Sam3Processor, Sam3Model
    except Exception:
        print('[setup] installing transformers from GitHub main for SAM3 support')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'git+https://github.com/huggingface/transformers'])
        from transformers import Sam3Processor, Sam3Model

print('[setup] ready')


In [ ]:
from pathlib import Path
import gc, json, math, os, random, time
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from sklearn.cluster import MiniBatchKMeans

import torch
from huggingface_hub import login
from transformers import Sam3Processor, Sam3Model

ImageFile.LOAD_TRUNCATED_IMAGES = True
SEED = 20260423
VERSION = 'sam3_sift_graph_v20260423'
random.seed(SEED)
np.random.seed(SEED)

WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT_DIR = WORK_DIR / 'animalclef_sam3_sift'
CROP_DIR = OUT_DIR / 'sam3_crops'
REPORT_DIR = OUT_DIR / 'reports'
SUB_DIR = OUT_DIR / 'submissions'
for d in [OUT_DIR, CROP_DIR, REPORT_DIR, SUB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
if DEVICE == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0))

# Read HF token from env or Kaggle Secret if available. Do not hardcode tokens in notebooks.
token = os.environ.get('HF_TOKEN')
if token is None:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        token = None
if token:
    login(token=token, add_to_git_credential=False)
    print('[hf] logged in from secret/env token')
else:
    print('[hf] no HF_TOKEN found; public/gated model access may fail')

def find_competition_root():
    candidates = [
        Path('/kaggle/input/animal-clef-2026'),
        Path('/kaggle/input/competitions/animal-clef-2026'),
        Path('/content/animal-clef-2026'),
        Path.cwd() / 'animal-clef-2026',
    ]
    for root in candidates:
        if (root / 'metadata.csv').exists() and (root / 'sample_submission.csv').exists():
            return root
    for base in [Path('/kaggle/input'), Path('/content'), Path.cwd()]:
        if not base.exists():
            continue
        for p in base.rglob('metadata.csv'):
            if (p.parent / 'sample_submission.csv').exists():
                return p.parent
    raise FileNotFoundError('AnimalCLEF2026 root not found')

DATA_ROOT = find_competition_root()
meta = pd.read_csv(DATA_ROOT / 'metadata.csv').reset_index(drop=True)
sample = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
meta['abs_path'] = meta['path'].map(lambda p: str((DATA_ROOT / str(p)).resolve()) if not Path(str(p)).is_absolute() else str(p))
print('DATA_ROOT:', DATA_ROOT)
print(meta.groupby(['dataset', 'split']).size())


In [ ]:
# Controls. Conservative defaults intentionally avoid giant components like the previous 570-image lynx cluster.
SAM3_MAX_SIDE = 896
SAM3_BATCH_SIZE = 8
SAM3_THRESHOLD = {
    'LynxID2025': 0.35,
    'SalamanderID2025': 0.30,
    'SeaTurtleID2022': 0.35,
    'TexasHornedLizards': 0.30,
}
SAM3_MASK_THRESHOLD = 0.50
PROMPT = {
    'LynxID2025': 'lynx animal',
    'SalamanderID2025': 'salamander animal',
    'SeaTurtleID2022': 'sea turtle head animal',
    'TexasHornedLizards': 'horned lizard animal',
}
SIFT_TOPK = {
    'LynxID2025': 60,
    'SalamanderID2025': 55,
    'SeaTurtleID2022': 50,
    'TexasHornedLizards': 55,
}
SIFT_EDGE_THR = {
    'LynxID2025': 0.34,
    'SalamanderID2025': 0.28,
    'SeaTurtleID2022': 0.30,
    'TexasHornedLizards': 0.28,
}
MIN_GOOD_MATCHES = {
    'LynxID2025': 24,
    'SalamanderID2025': 16,
    'SeaTurtleID2022': 18,
    'TexasHornedLizards': 16,
}
MAX_COMPONENT_SIZE = {
    'LynxID2025': 80,
    'SalamanderID2025': 25,
    'SeaTurtleID2022': 35,
    'TexasHornedLizards': 30,
}
BOVW_WORDS = {
    'LynxID2025': 384,
    'SalamanderID2025': 384,
    'SeaTurtleID2022': 320,
    'TexasHornedLizards': 256,
}
MAX_DESC_PER_IMAGE = 1800
MAX_DESC_SAMPLE_PER_IMAGE = 400
print('version:', VERSION)


In [ ]:
print('[model] loading SAM3')
sam_model = Sam3Model.from_pretrained('facebook/sam3').to(DEVICE).eval()
sam_processor = Sam3Processor.from_pretrained('facebook/sam3')
print('[model] SAM3 ready')

def load_resized_rgb(path, max_side=SAM3_MAX_SIDE):
    img = Image.open(path).convert('RGB')
    w, h = img.size
    scale = min(1.0, max_side / max(w, h))
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
    return img

def choose_mask(masks, scores, image_size):
    if masks is None or len(masks) == 0:
        return None
    arr = masks.detach().cpu().numpy().astype(bool)
    scores_np = scores.detach().cpu().numpy() if scores is not None else np.ones(len(arr), dtype='float32')
    h, w = arr.shape[-2:]
    img_area = float(h * w)
    best = None
    best_score = -1
    for mask, score in zip(arr, scores_np):
        area = float(mask.sum())
        frac = area / img_area
        if frac < 0.005 or frac > 0.92:
            continue
        ys, xs = np.where(mask)
        if len(xs) == 0:
            continue
        cx = (xs.mean() / max(w - 1, 1)) - 0.5
        cy = (ys.mean() / max(h - 1, 1)) - 0.5
        centrality = 1.0 - min(1.0, math.sqrt(cx * cx + cy * cy) / 0.70)
        rank_score = float(score) * 0.65 + min(frac / 0.30, 1.0) * 0.20 + centrality * 0.15
        if rank_score > best_score:
            best_score = rank_score
            best = mask
    return best

def segment_and_crop(path, species):
    path = Path(path)
    out_dir = CROP_DIR / species
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / (path.stem + '_sam3.jpg')
    if out_path.exists():
        return str(out_path), True
    image = load_resized_rgb(path)
    prompt = PROMPT[species]
    try:
        inputs = sam_processor(images=image, text=prompt, return_tensors='pt').to(DEVICE)
        with torch.inference_mode():
            outputs = sam_model(**inputs)
        results = sam_processor.post_process_instance_segmentation(
            outputs,
            threshold=SAM3_THRESHOLD[species],
            mask_threshold=SAM3_MASK_THRESHOLD,
            target_sizes=inputs.get('original_sizes').tolist(),
        )[0]
        mask = choose_mask(results.get('masks'), results.get('scores'), image.size)
    except Exception as e:
        mask = None
    img = np.array(image)
    if mask is None:
        crop = img
        ok = False
    else:
        ys, xs = np.where(mask)
        if len(xs) == 0:
            crop = img
            ok = False
        else:
            h, w = mask.shape
            margin = int(0.08 * max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1)) + 8
            x0, x1 = max(0, xs.min() - margin), min(w, xs.max() + margin + 1)
            y0, y1 = max(0, ys.min() - margin), min(h, ys.max() + margin + 1)
            mask3 = mask[..., None]
            fg = np.where(mask3, img, 0)
            crop = fg[y0:y1, x0:x1]
            ok = True
    if min(crop.shape[:2]) < 64:
        crop = img
        ok = False
    cv2.imwrite(str(out_path), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 94])
    return str(out_path), ok

def save_crop_from_mask(path, species, image, mask):
    path = Path(path)
    out_dir = CROP_DIR / species
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / (path.stem + '_sam3.jpg')
    if out_path.exists():
        return str(out_path), True
    img = np.array(image)
    if mask is None:
        crop = img
        ok = False
    else:
        ys, xs = np.where(mask)
        if len(xs) == 0:
            crop = img
            ok = False
        else:
            h, w = mask.shape
            margin = int(0.08 * max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1)) + 8
            x0, x1 = max(0, xs.min() - margin), min(w, xs.max() + margin + 1)
            y0, y1 = max(0, ys.min() - margin), min(h, ys.max() + margin + 1)
            crop = np.where(mask[..., None], img, 0)[y0:y1, x0:x1]
            ok = True
    if min(crop.shape[:2]) < 64:
        crop = img
        ok = False
    cv2.imwrite(str(out_path), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 94])
    return str(out_path), ok

def segment_and_crop_batch(paths, species, batch_size=SAM3_BATCH_SIZE):
    paths = [str(p) for p in paths]
    outputs = [None] * len(paths)
    pending_slots, pending_paths, pending_images = [], [], []
    for slot, path in enumerate(paths):
        p = Path(path)
        out_path = CROP_DIR / species / (p.stem + '_sam3.jpg')
        if out_path.exists():
            outputs[slot] = (str(out_path), True)
        else:
            pending_slots.append(slot)
            pending_paths.append(p)
            pending_images.append(load_resized_rgb(p))
    prompt = PROMPT[species]
    for start in tqdm(range(0, len(pending_paths), batch_size), desc=f'SAM3 batches {species}'):
        batch_slots = pending_slots[start:start + batch_size]
        batch_paths = pending_paths[start:start + batch_size]
        batch_images = pending_images[start:start + batch_size]
        masks = [None] * len(batch_images)
        try:
            inputs = sam_processor(images=batch_images, text=[prompt] * len(batch_images), return_tensors='pt').to(DEVICE)
            with torch.inference_mode():
                if DEVICE == 'cuda':
                    with torch.autocast(device_type='cuda', dtype=torch.float16):
                        sam_outputs = sam_model(**inputs)
                else:
                    sam_outputs = sam_model(**inputs)
            results = sam_processor.post_process_instance_segmentation(
                sam_outputs,
                threshold=SAM3_THRESHOLD[species],
                mask_threshold=SAM3_MASK_THRESHOLD,
                target_sizes=inputs.get('original_sizes').tolist(),
            )
            for k, result in enumerate(results):
                masks[k] = choose_mask(result.get('masks'), result.get('scores'), batch_images[k].size)
        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and DEVICE == 'cuda' and batch_size > 1:
                torch.cuda.empty_cache()
                print(f'[sam3] OOM at batch_size={batch_size}; retrying this batch one image at a time')
                single_results = segment_and_crop_batch([str(p) for p in batch_paths], species, batch_size=1)
                for slot, result in zip(batch_slots, single_results):
                    outputs[slot] = result
                continue
            print('[sam3] batch failed:', repr(e))
        except Exception as e:
            print('[sam3] batch failed:', repr(e))
        for slot, p, img, mask in zip(batch_slots, batch_paths, batch_images, masks):
            outputs[slot] = save_crop_from_mask(p, species, img, mask)
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
    return outputs


In [ ]:
sift = cv2.SIFT_create(nfeatures=MAX_DESC_PER_IMAGE, contrastThreshold=0.018, edgeThreshold=12, sigma=1.2)

def rootsift(desc):
    if desc is None or len(desc) == 0:
        return None
    desc = desc.astype('float32')
    desc /= (desc.sum(axis=1, keepdims=True) + 1e-12)
    desc = np.sqrt(desc)
    desc /= (np.linalg.norm(desc, axis=1, keepdims=True) + 1e-12)
    return desc.astype('float32')

def extract_sift(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return [], None
    # CLAHE helps salamander spots and turtle scutes survive variable exposure.
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)
    kp, desc = sift.detectAndCompute(img, None)
    desc = rootsift(desc)
    if desc is not None and len(desc) > MAX_DESC_PER_IMAGE:
        desc = desc[:MAX_DESC_PER_IMAGE]
        kp = kp[:MAX_DESC_PER_IMAGE]
    return kp, desc

def build_bovw(descs, species):
    rng = np.random.default_rng(SEED + sum(map(ord, species)))
    samples = []
    for d in descs:
        if d is None or len(d) == 0:
            continue
        take = min(MAX_DESC_SAMPLE_PER_IMAGE, len(d))
        idx = rng.choice(len(d), size=take, replace=False)
        samples.append(d[idx])
    if not samples:
        return np.eye(len(descs), dtype='float32')
    train_desc = np.vstack(samples).astype('float32')
    n_words = min(BOVW_WORDS[species], max(32, len(train_desc) // 8))
    print(f'[bovw] {species}: train_desc={train_desc.shape}, words={n_words}')
    kmeans = MiniBatchKMeans(n_clusters=n_words, random_state=SEED, batch_size=8192, n_init=3, max_iter=120)
    kmeans.fit(train_desc)
    hist = np.zeros((len(descs), n_words), dtype='float32')
    for i, d in enumerate(tqdm(descs, desc=f'bovw hist {species}')):
        if d is None or len(d) == 0:
            continue
        words = kmeans.predict(d.astype('float32'))
        h = np.bincount(words, minlength=n_words).astype('float32')
        hist[i] = h
    df = (hist > 0).sum(axis=0)
    idf = np.log((1 + len(descs)) / (1 + df)) + 1.0
    hist *= idf[None, :]
    hist /= (np.linalg.norm(hist, axis=1, keepdims=True) + 1e-12)
    return hist

bf = cv2.BFMatcher(cv2.NORM_L2)

def sift_pair_score(kp1, desc1, kp2, desc2, coarse_score):
    if desc1 is None or desc2 is None or len(desc1) < 8 or len(desc2) < 8:
        return 0.0, 0, 0.0
    try:
        matches = bf.knnMatch(desc1, desc2, k=2)
    except Exception:
        return 0.0, 0, 0.0
    good = []
    for pair in matches:
        if len(pair) < 2:
            continue
        m, n = pair
        if m.distance < 0.76 * n.distance:
            good.append(m)
    good_n = len(good)
    inlier_ratio = 0.0
    if good_n >= 8:
        pts1 = np.float32([kp1[m.queryIdx].pt for m in good])
        pts2 = np.float32([kp2[m.trainIdx].pt for m in good])
        try:
            _, mask = cv2.estimateAffinePartial2D(pts1, pts2, method=cv2.RANSAC, ransacReprojThreshold=5.0, maxIters=1000)
            if mask is not None:
                inlier_ratio = float(mask.mean())
        except Exception:
            inlier_ratio = 0.0
    match_score = min(1.0, good_n / 80.0)
    score = 0.62 * match_score + 0.23 * inlier_ratio + 0.15 * float(max(coarse_score, 0.0))
    return float(score), int(good_n), float(inlier_ratio)


In [ ]:
class DSU:
    def __init__(self, n):
        self.p = list(range(n))
        self.sz = [1] * n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b, max_size=None):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return True
        if max_size is not None and self.sz[ra] + self.sz[rb] > max_size:
            return False
        if self.sz[ra] < self.sz[rb]:
            ra, rb = rb, ra
        self.p[rb] = ra
        self.sz[ra] += self.sz[rb]
        return True

def mutual_candidate_edges(coarse, topk):
    n = coarse.shape[0]
    np.fill_diagonal(coarse, -1.0)
    k = min(topk, max(1, n - 1))
    nbr = np.argpartition(-coarse, kth=np.arange(k), axis=1)[:, :k]
    nbr_sets = [set(row.tolist()) for row in nbr]
    edges = []
    for i in range(n):
        for j in nbr[i]:
            j = int(j)
            if j <= i:
                continue
            if i in nbr_sets[j]:
                edges.append((float(coarse[i, j]), i, j))
    edges.sort(reverse=True)
    return edges

def cluster_species(species):
    rows = meta[(meta['dataset'] == species) & (meta['split'] == 'test')].copy().reset_index(drop=True)
    print(f'\n[species] {species}: {len(rows)} test images')
    crop_results = segment_and_crop_batch(rows['abs_path'].tolist(), species)
    crop_paths = [x[0] for x in crop_results]
    mask_ok = [x[1] for x in crop_results]
    print(f'[sam3] {species}: masks ok {sum(mask_ok)}/{len(mask_ok)}')

    kps, descs = [], []
    for path in tqdm(crop_paths, desc=f'SIFT {species}'):
        kp, desc = extract_sift(path)
        kps.append(kp)
        descs.append(desc)
    desc_counts = [0 if d is None else len(d) for d in descs]
    print(f'[sift] {species}: desc median={np.median(desc_counts):.1f}, mean={np.mean(desc_counts):.1f}')

    hist = build_bovw(descs, species)
    coarse = hist @ hist.T
    candidate_edges = mutual_candidate_edges(coarse.copy(), SIFT_TOPK[species])
    print(f'[edges] {species}: candidates={len(candidate_edges)}')

    scored_edges = []
    for coarse_score, i, j in tqdm(candidate_edges, desc=f'pair match {species}'):
        score, good_n, inlier = sift_pair_score(kps[i], descs[i], kps[j], descs[j], coarse_score)
        if good_n >= MIN_GOOD_MATCHES[species] or score >= SIFT_EDGE_THR[species] - 0.03:
            scored_edges.append((score, good_n, inlier, i, j))
    scored_edges.sort(reverse=True)

    dsu = DSU(len(rows))
    used_edges = 0
    for score, good_n, inlier, i, j in scored_edges:
        if score < SIFT_EDGE_THR[species]:
            break
        if good_n < MIN_GOOD_MATCHES[species]:
            continue
        if dsu.union(i, j, max_size=MAX_COMPONENT_SIZE[species]):
            used_edges += 1

    root_to_id = {}
    clusters = []
    for i in range(len(rows)):
        r = dsu.find(i)
        if r not in root_to_id:
            root_to_id[r] = len(root_to_id)
        clusters.append(f'cluster_{species}_{root_to_id[r]}')
    part = pd.DataFrame({'image_id': rows['image_id'].to_numpy(), 'cluster': clusters})
    vc = part['cluster'].value_counts()
    report = {
        'species': species,
        'n_images': int(len(rows)),
        'sam3_masks_ok': int(sum(mask_ok)),
        'median_sift_desc': float(np.median(desc_counts)),
        'mean_sift_desc': float(np.mean(desc_counts)),
        'candidate_edges': int(len(candidate_edges)),
        'scored_edges_kept': int(len(scored_edges)),
        'used_edges': int(used_edges),
        'n_clusters': int(part['cluster'].nunique()),
        'max_cluster_size': int(vc.max()),
        'singleton_clusters': int((vc == 1).sum()),
    }
    print('[report]', report)
    return part, report

parts = []
reports = []
for species in ['LynxID2025', 'SalamanderID2025', 'SeaTurtleID2022', 'TexasHornedLizards']:
    part, report = cluster_species(species)
    parts.append(part)
    reports.append(report)

sub = pd.concat(parts, ignore_index=True)
sub = sample[['image_id']].merge(sub, on='image_id', how='left')
assert len(sub) == len(sample)
assert sub['cluster'].notna().all()
assert list(sub.columns) == ['image_id', 'cluster']

versioned = SUB_DIR / f'submission_{VERSION}.csv'
sub.to_csv(versioned, index=False)
sub.to_csv(WORK_DIR / 'submission.csv', index=False)
summary = {'version': VERSION, 'submission': str(versioned), 'reports': reports}
(REPORT_DIR / f'run_summary_{VERSION}.json').write_text(json.dumps(summary, indent=2))
print('wrote', versioned)
print('wrote', WORK_DIR / 'submission.csv')
print(sub.head())
print(json.dumps(summary, indent=2)[:5000])
